<!-- notebook-header -->
# Algebra Linear II: Matrizes e Transformacoes

**Modulo:** 00 - Matematica  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Matrizes, transformacoes lineares, sistemas, autovalores, PCA, SVD e recomendacao.


# Algebra Linear II: Matrizes e Transformacoes

Estruturando dados multidimensionais — transformacoes lineares, decomposicoes e suas aplicacoes em ML

---

## Pre-requisitos e Conexoes

| Topico | Notebook | Tipo |
|--------|----------|------|
| Vetores e Operacoes | 0.2 | Pre-requisito |
| Matrizes (este) | 0.3 | Atual |
| Calculo Multivariado | 0.4 | Proximo |
| Autovalores/PCA Detalhado | 0.5 | Proximo |
| Regressao Linear | 1.2 | Aplicacao |
| Redes Neurais | 2.1 | Aplicacao |

---

## Indice

1. Introducao e Setup
2. Tipos de Matrizes
3. Operacoes Basicas
4. Multiplicacao (A@B)
5. Transformacoes Lineares em 2D
6. Sistemas Lineares Ax=b
7. Determinante
8. Rank de Matriz
9. Inversa e Pseudo-inversa
10. Autovalores/Autovetores
11. PCA do Zero
12. SVD
13. Compressao com SVD
14. Recomendacao com SVD
15. Matrizes em ML
16. Exercicios
17. Erros Comuns
18. Resumo Final

---
**Pre-requisitos**: Algebra Linear I (0.2) | **Tempo**: 12-14h | **Proximo**: Calculo I (0.4)

## Intuicao: O que sao Matrizes?

Pense em uma matriz como uma **tabela de numeros** ou como uma **funcao que transforma vetores**.

**Analogia do Mundo Real:**
- Uma imagem em preto-e-branco de 28x28 pixels e uma matriz de valores entre 0 e 255
- Uma tabela de planilha (linhas=produtos, colunas=lojas, valores=vendas) e uma matriz
- Um neuronio de rede neural transforma um vetor de entrada em outro usando uma matriz de pesos

**Em ML, uma matriz e um dataset:**
Se voce tem 100 fotos de 28x28 pixels cada uma, voce pode organizar como:
- Matriz 100 x 784 (100 exemplos, 784 pixels cada)
- Cada linha e um vetor (uma foto)
- Operacoes matriciais processam TAREFA DO ALUNOS os exemplos de uma vez (vetorizacao)

**Por que matrizes importam mais que vetores:**
- Dados reais vem em lotes: muitos exemplos simultaneos
- Redes neurais empilham transformacoes: uma matriz sobre outra
- Decomposicoes de matrizes (SVD, autovalores) desvendam estrutura oculta dos dados

## 1. Introdução

Matrizes transformam dados. Em ML: **Y = X @ W** (forward pass).
- X: dados (n_samples × n_features)
- W: pesos (n_features × n_output)
- Y: saída

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print('Imports OK')

## 2. Tipos de Matrizes

Antes de operar com matrizes, precisamos reconhecer seus tipos. Cada tipo tem propriedades que simplificam calculos e aparecem repetidamente em ML.

**Analogia:** Assim como na quimica existem elementos com propriedades especiais (gases nobres, metais), em algebra linear existem matrizes com propriedades especiais que facilitam a vida.

| Tipo | Propriedade | Por que importa em ML |
|------|------------|----------------------|
| **Identidade I** | I @ A = A | Inicializacao de pesos, termo neutro |
| **Diagonal** | Inversa trivial: D^(-1) = 1/d_ii | Escalar features independentemente |
| **Simetrica** | A = A^T | Matrizes de covariancia, kernels |
| **Ortogonal** | A^T @ A = I | Preserva normas, rotacoes, inicializacao estavel |
| **Positivo-definida** | x^T A x > 0 | Covariancia, Hessiano em minimos |
| **Esparsa** | Maioria zeros | Dados de texto (bag-of-words), grafos |

In [ ]:
# === Tipos de Matrizes: Exemplos e Visualizacao ===

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Matrizes para visualizar
matrices = {
    'Identidade (3x3)': np.eye(3),
    'Diagonal': np.diag([2, 3, 5]),
    'Simetrica': np.array([[4, 2, 1], [2, 5, 3], [1, 3, 6]], dtype=float),
    'Ortogonal\n(Rotacao 45°)': np.array([[np.cos(np.pi/4), -np.sin(np.pi/4), 0],
                                          [np.sin(np.pi/4), np.cos(np.pi/4), 0],
                                          [0, 0, 1]]),
    'Positivo-Definida': np.array([[4, 2, 0], [2, 5, 1], [0, 1, 3]], dtype=float),
    'Esparsa\n(Bag-of-Words)': np.array([[0, 3, 0], [1, 0, 0], [0, 0, 2]], dtype=float),
}

for ax, (name, M) in zip(axes.flat, matrices.items()):
    im = ax.imshow(M, cmap='RdBu_r', vmin=-5, vmax=5, aspect='equal')
    ax.set_title(name, fontsize=11, fontweight='bold')
    # Anotar valores
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f'{M[i,j]:.1f}', ha='center', va='center',
                    fontsize=10, color='white' if abs(M[i,j]) > 2.5 else 'black')
    ax.set_xticks(range(M.shape[1]))
    ax.set_yticks(range(M.shape[0]))

fig.colorbar(im, ax=axes, shrink=0.6, label='Valor')
fig.suptitle('Tipos de Matrizes: Padroes Visuais', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# Verificacoes numericas
S = matrices['Simetrica']
Q = matrices['Ortogonal\n(Rotacao 45°)']
PD = matrices['Positivo-Definida']

print("=== Verificacoes ===")
print(f"Simetrica? A == A^T: {np.allclose(S, S.T)}")
print(f"Ortogonal? Q^T @ Q == I: {np.allclose(Q.T @ Q, np.eye(3))}")
print(f"Positivo-definida? Todos autovalores > 0: {np.all(np.linalg.eigvalsh(PD) > 0)}")
print(f"  Autovalores de PD: {np.linalg.eigvalsh(PD).round(2)}")

### O que observar

- Verificar a forma (shape) da matriz resultante
- Valores numéricos e tipos de dados
- Propriedades de matrizes especiais (diagonal, simétrica, etc)

## 3. Operacoes Basicas

Operacoes com matrizes sao a base de tudo que vem depois. Em ML, cada uma tem um papel:

- **Adicao (A + B):** Combinar informacoes. Ex: somar bias a cada exemplo do batch
- **Escalar (alpha * A):** Learning rate multiplicando gradientes
- **Transposicao (A^T):** Trocar linhas por colunas. Essencial para formulas como X^T @ X (Gram matrix)

**Regra importante:** Dimensoes devem ser compativeis. A + B so funciona se A e B tem o mesmo shape. Em NumPy, broadcasting pode "expandir" dimensoes automaticamente (cuidado: fonte comum de bugs).

#### por que em ml

- Dados em ML são matrizes: cada linha é um exemplo, cada coluna é uma feature
- Vetorização: processar 1000 exemplos com uma multiplicação matricial é muito mais rápido que loop Python
- GPU otimizado para operações matriciais

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print('A+B:\n', A+B)
print('2*A:\n', 2*A)
print('A^T:\n', A.T)

### O que observar

- Verificar a forma (shape) da matriz resultante
- Valores numéricos e tipos de dados
- Propriedades de matrizes especiais (diagonal, simétrica, etc)

### O que concluir

- Matrizes permitem processar múltiplos vetores simultaneamente
- Operações matriciais são a base da vetorização em ML- Formato matricial é essencial para computação eficiente

## 4. Multiplicacao de Matrizes: A @ B

**Intuicao:** Multiplicar matrizes e como encadear transformacoes. Se A roda 30 graus e B escala 2x, entao A @ B faz as duas coisas em sequencia. Em ML, cada camada de uma rede neural e uma multiplicacao matricial -- a rede inteira e uma cadeia de multiplicacoes.

**Definicao:** (A @ B)_ij = soma_k (a_ik * b_kj)

Cada elemento do resultado e um **produto interno** entre uma linha de A e uma coluna de B.

**Regra de compatibilidade:**
- A tem shape (m, p) e B tem shape (p, n) --> resultado tem shape (m, n)
- A dimensao "interna" (p) deve coincidir
- **Nao e comutativo:** A @ B != B @ A em geral

**Em ML (Forward Pass):**
```
Y = X @ W + b
X: (batch_size, n_features)    -- dados
W: (n_features, n_output)      -- pesos aprendiveis
b: (n_output,)                 -- bias (broadcasting)
Y: (batch_size, n_output)      -- predicoes
```

In [ ]:
# === Multiplicacao de Matrizes: Visualizacao ===

A = np.array([[1, 2, 3], [4, 5, 6]], dtype=float)
B = np.array([[7, 8], [9, 10], [11, 12]], dtype=float)
C = A @ B

fig, axes = plt.subplots(1, 4, figsize=(16, 4),
                          gridspec_kw={'width_ratios': [3, 2, 0.3, 2]})

# Painel 1: Visualizar A
def show_matrix(ax, M, title, cmap='Blues'):
    im = ax.imshow(M, cmap=cmap, aspect='equal')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f'{M[i,j]:.0f}', ha='center', va='center',
                    fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticks(range(M.shape[1]))
    ax.set_yticks(range(M.shape[0]))
    ax.set_xticklabels([f'c{j}' for j in range(M.shape[1])])
    ax.set_yticklabels([f'r{i}' for i in range(M.shape[0])])

show_matrix(axes[0], A, f'A ({A.shape[0]}x{A.shape[1]})', 'Blues')
show_matrix(axes[1], B, f'B ({B.shape[0]}x{B.shape[1]})', 'Greens')

# Simbolo @
axes[2].text(0.5, 0.5, '@', fontsize=30, ha='center', va='center',
             fontweight='bold', transform=axes[2].transAxes)
axes[2].axis('off')

show_matrix(axes[3], C, f'C = A@B ({C.shape[0]}x{C.shape[1]})', 'Oranges')

fig.suptitle('Multiplicacao Matricial: A(2x3) @ B(3x2) = C(2x2)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Demonstrar elemento especifico
print(f"=== Como calcular C[0,0] ===")
print(f"Linha 0 de A: {A[0]}")
print(f"Coluna 0 de B: {B[:, 0]}")
print(f"Produto interno: {A[0]} . {B[:, 0]} = {A[0,0]}*{B[0,0]} + {A[0,1]}*{B[1,0]} + {A[0,2]}*{B[2,0]} = {C[0,0]:.0f}")

# Forward pass: demonstracao pratica
print(f"\n=== Forward Pass de Rede Neural ===")
X = np.array([[1.0, 0.5], [2.0, 1.5], [0.5, 2.0]])  # 3 exemplos, 2 features
W = np.array([[0.3, -0.2, 0.5], [0.1, 0.4, -0.1]])   # 2 features -> 3 neuronios
b = np.array([0.1, -0.2, 0.3])                         # bias
Y = X @ W + b
print(f"X: {X.shape} (batch=3, features=2)")
print(f"W: {W.shape} (features=2, neuronios=3)")
print(f"Y = X@W+b: {Y.shape} (batch=3, neuronios=3)")
print(f"Y =\n{Y.round(2)}")
print(f"\nNota: SEM multiplicacao matricial, precisariamos de um loop para cada exemplo!")

### O que concluir

- Matrizes permitem processar múltiplos vetores simultaneamente
- Operações matriciais são a base da vetorização em ML- Formato matricial é essencial para computação eficiente

## 5. Transformacoes Lineares em 2D

**Intuicao:** Uma matriz 2x2 e uma "receita de transformacao". Ela pega cada ponto do plano e move para um novo lugar. Para entender o que a matriz faz, basta ver para onde ela envia os vetores base e1=(1,0) e e2=(0,1).

**Tipos fundamentais:**

| Transformacao | Efeito | Exemplo em ML |
|---------------|--------|---------------|
| **Rotacao** | Gira pontos | Data augmentation em imagens |
| **Escala** | Estica/comprime | Normalizacao de features |
| **Cisalhamento** | Inclina | Distorcoes geometricas |
| **Reflexao** | Espelha | Data augmentation (flip) |

**Por que importa:** Em redes neurais, cada camada linear aplica uma transformacao desse tipo (combinada). Treinar a rede e encontrar as transformacoes certas para separar classes.

In [ ]:
# === Transformacoes Lineares 2D: 6 Tipos ===

# Gerar nuvem de pontos (circulo unitario + pontos aleatorios)
theta = np.linspace(0, 2*np.pi, 100)
circle = np.array([np.cos(theta), np.sin(theta)])  # circulo unitario
np.random.seed(42)
points = np.random.randn(2, 50) * 0.4  # pontos internos
all_points = np.hstack([circle, points])

# 6 transformacoes fundamentais
angle = np.pi/4  # 45 graus
transforms = {
    'Identidade\nI': np.eye(2),
    'Rotacao 45°\nR(θ)': np.array([[np.cos(angle), -np.sin(angle)],
                                    [np.sin(angle), np.cos(angle)]]),
    'Escala\ndiag(2, 0.5)': np.diag([2.0, 0.5]),
    'Cisalhamento\n(shear)': np.array([[1, 0.8], [0, 1]]),
    'Reflexao X\n(flip y)': np.array([[1, 0], [0, -1]]),
    'Reflexao y=x\n(swap axes)': np.array([[0, 1], [1, 0]]),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, (name, M) in zip(axes.flat, transforms.items()):
    transformed = M @ all_points

    # Plotar original (cinza) e transformado (azul)
    ax.plot(circle[0], circle[1], 'gray', alpha=0.3, linewidth=1, label='Original')
    ax.scatter(points[0], points[1], c='gray', s=8, alpha=0.3)

    t_circle = M @ circle
    t_points = M @ points
    ax.plot(t_circle[0], t_circle[1], 'steelblue', linewidth=2, label='Transformado')
    ax.scatter(t_points[0], t_points[1], c='steelblue', s=10, alpha=0.6)

    # Vetores base transformados
    e1_new = M @ np.array([1, 0])
    e2_new = M @ np.array([0, 1])
    ax.annotate('', xy=e1_new, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax.annotate('', xy=e2_new, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='green', lw=2))
    ax.plot(0, 0, 'ko', markersize=4)

    det = np.linalg.det(M)
    ax.set_title(f'{name}\ndet={det:.2f}', fontsize=10, fontweight='bold')
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)

axes[0, 0].legend(fontsize=8, loc='upper left')
fig.suptitle('6 Transformacoes Lineares Fundamentais\n(vermelho=e1 transformado, verde=e2 transformado)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Legenda: setas vermelhas = para onde (1,0) vai, setas verdes = para onde (0,1) vai")
print("O determinante indica o fator de escala de area (negativo = reflexao)")

#### conexao com

- **Vetores (0.2)**: Matrizes são conjuntos de vetores (linhas ou colunas)- **Transformações lineares**: Multiplicação Ax transforma vetores

### Erro

**Confundir multiplicação elemento-a-elemento com matricial:**

```python# ❌ ERRADO: elemento-a-elementoA * B  # com numpy arrays# ✓ CORRETO: multiplicação matricialA @ B  # ou np.matmul(A, B)```

#### conexao com

- **Vetores (0.2)**: Matrizes são conjuntos de vetores (linhas ou colunas)- **Transformações lineares**: Multiplicação Ax transforma vetores

### O que observar

- Padrões nos autovalores (crescimento, decrescimento)- Relação entre autovalores e comportamento da transformação
- Qual autovalor é dominante?

#### por que em ml

- Regressão linear: solução analítica é (X^T X)^-1 X^T y
- Entender inversão de matrizes é essencial para entender regressão

**O Que Observar e Concluir**

**Observacoes dos 6 Graficos:**
1. **Identidade**: Os pontos nao mudam (I @ x = x). Setas vermelha e verde apontam para (1,0) e (0,1)
2. **Rotacao 45 graus**: A nuvem de pontos roda, mas conserva distancias (det = 1.00, sem escala)
3. **Escala**: O circulo vira elipse -- um eixo estica (2x), outro comprime (0.5x). det = 1.00 (area preservada!)
4. **Cisalhamento**: A nuvem "inclina" para um lado, como empurrar um baralho de cartas lateralmente
5. **Reflexao X**: Os pontos espelham no eixo x (coordenada y inverte). det = -1 (reflexao = sinal negativo)
6. **Reflexao y=x**: Coordenadas x e y trocam. det = -1

**O que concluir:**
- Cada coluna da matriz diz para onde vai o vetor base correspondente (e1 -> coluna 1, e2 -> coluna 2)
- |det| > 1: expansao de area. |det| < 1: compressao. |det| = 1: area preservada. det < 0: reflexao
- Em redes neurais, cada camada linear faz uma combinacao dessas transformacoes. Treinar a rede e aprender QUAIS transformacoes separar as classes

## 6. Sistemas Lineares: Ax = b

**Intuicao:** Resolver Ax = b e encontrar o vetor x que, quando transformado pela matriz A, produz b. E como perguntar: "qual entrada produz esta saida?"

**Em ML, isso aparece em:**
- **Regressao linear:** X @ beta = y. Queremos encontrar beta (pesos) que melhor explica os dados
- **Minimos quadrados:** Quando o sistema e sobredeterminado (mais equacoes que incognitas), encontramos a solucao que minimiza o erro quadratico
- **Normal equation:** beta = (X^T @ X)^(-1) @ X^T @ y

**Tres cenarios:**

| Cenario | Condicao | Solucao | Em ML |
|---------|----------|---------|-------|
| **Unica** | det(A) != 0 | x = A^(-1) @ b | Sistema determinado |
| **Infinitas** | rank(A) < n | Infinitas solucoes | Features redundantes |
| **Nenhuma** | Inconsistente | Minimos quadrados | Caso tipico (mais dados que features) |

### Erro

**Confundir multiplicação elemento-a-elemento com matricial:**

```python# ❌ ERRADO: elemento-a-elementoA * B  # com numpy arrays# ✓ CORRETO: multiplicação matricialA @ B  # ou np.matmul(A, B)```

In [ ]:
A = np.array([[3, 1], [1, 2]], dtype=float)
b = np.array([9, 8], dtype=float)

print('det(A) =', np.linalg.det(A))
x = np.linalg.solve(A, b)
print('Solução x =', x)
print('Verificação A@x =', A@x)

### O que observar

- Padrões nos autovalores (crescimento, decrescimento)- Relação entre autovalores e comportamento da transformação
- Qual autovalor é dominante?

#### por que em ml

- Dados em ML são matrizes: cada linha é um exemplo, cada coluna é uma feature
- Vetorização: processar 1000 exemplos com uma multiplicação matricial é muito mais rápido que loop Python
- GPU otimizado para operações matriciais

## 7. Determinante

**Intuicao:** O determinante mede quanto uma matriz "estica" ou "comprime" areas (em 2D) ou volumes (em 3D). E como o fator de zoom de uma transformacao.

**Propriedades-chave:**
- det(A) > 0: transformacao preserva orientacao
- det(A) < 0: transformacao inverte orientacao (reflexao)
- det(A) = 0: a matriz "achata" o espaco (perde uma dimensao) -- nao invertivel!
- |det(A)| = fator de escala de area/volume

**Por que importa em ML:**
- Se det(X^T @ X) = 0 (ou proximo de 0), regressao linear nao tem solucao unica (multicolinearidade)
- O Jacobiano (matriz de derivadas parciais) usa determinante para medir mudancas de volume em distribuicoes
- Em VAEs e Normalizing Flows, o log|det(J)| aparece explicitamente na funcao de custo

In [ ]:
# === Determinante: Visualizacao Geometrica ===

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Quadrado unitario
square = np.array([[0, 1, 1, 0, 0],
                   [0, 0, 1, 1, 0]])

matrices_det = {
    'det=3 (expansao)': np.array([[2, 0.5], [0, 1.5]]),
    'det=0 (singular!)': np.array([[1, 2], [0.5, 1]]),
    'det=-2 (reflexao)': np.array([[-1, 0.5], [0.5, 2]]),
}

for ax, (name, M) in zip(axes, matrices_det.items()):
    det = np.linalg.det(M)
    transformed = M @ square

    # Quadrado original
    ax.fill(square[0], square[1], alpha=0.3, color='lightblue', label=f'Original (area=1)')
    ax.plot(square[0], square[1], 'b-', linewidth=1.5)

    # Paralelogramo transformado
    color = 'red' if abs(det) < 0.01 else ('orange' if det < 0 else 'green')
    ax.fill(transformed[0], transformed[1], alpha=0.3, color=color,
            label=f'Transformado (area={abs(det):.1f})')
    ax.plot(transformed[0], transformed[1], color=color, linewidth=2)

    # Vetores base transformados
    e1 = M @ np.array([1, 0])
    e2 = M @ np.array([0, 1])
    ax.annotate('', xy=e1, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax.annotate('', xy=e2, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='darkgreen', lw=2))

    ax.set_title(f'{name}\ndet = {det:.2f}', fontsize=11, fontweight='bold')
    ax.set_xlim(-2.5, 3.5)
    ax.set_ylim(-1.5, 3.5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='upper left')
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)

fig.suptitle('Determinante = Fator de Escala de Area', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("det > 0: preserva orientacao (verde)")
print("det < 0: inverte orientacao/reflexao (laranja)")
print("det = 0: colapsa em linha -- NAO invertivel! (vermelho)")

### O que concluir

- Autovalores medem a 'força' de uma transformação em cada direção
- Autovetores definem as direções principais de transformação
- Decomposição em autovalores revela estrutura da matriz

**O Que Observar e Concluir**

**Geometria do Determinante nos 3 paineis:**
- **det = 3 (verde):** O quadrado unitario (area = 1) vira um paralelogramo de area 3. A transformacao "expande" o espaco
- **det = 0 (vermelho):** O quadrado colapsa em uma LINHA. Toda a informacao 2D e "achatada" em 1D. A matriz e singular (nao invertivel) -- impossivel desfazer a transformacao porque informacao foi perdida
- **det = -2 (laranja):** Area = 2 (expansao), mas com reflexao (orientacao invertida)

**Aplicacao em ML:**
- Se det(X^T @ X) esta proximo de 0, a regressao linear e numericamente instavel (multicolinearidade). Regularizacao L2 (Ridge) adiciona lambda * I, garantindo det > 0
- Em Normalizing Flows, o modelo precisa calcular log|det(J)| a cada passo -- por isso usam transformacoes com Jacobiano triangular (det = produto da diagonal, rapido de calcular)

#### por que em ml

- Redes neurais: cada camada é y = Wx + b (multiplicação matricial)- Backpropagation: multiplicação por transposta da matriz de pesos
- Toda otimização em deep learning é matricial

## 8. Rank: Dimensao Efetiva dos Dados

**Intuicao:** O rank de uma matriz e o numero de direcoes "verdadeiramente independentes" que ela contem. Se voce tem 100 features mas rank = 3, seus dados vivem num subespaco 3D (as outras 97 dimensoes sao redundantes).

**Analogia:** Imagine 3 termometros: um em Celsius, outro em Fahrenheit, outro em Kelvin. Sao 3 features, mas rank = 1 porque todas medem a mesma coisa (temperatura).

**Definicao:** rank(A) = numero de linhas (ou colunas) linearmente independentes

**Por que importa em ML:**
- **Multicolinearidade:** Se rank(X) < num_features, regressao linear tem infinitas solucoes (o sistema e indeterminado)
- **Compressao:** Se rank(dados) << num_features, podemos reduzir dimensionalidade sem perder informacao (PCA)
- **Regularizacao:** L2/Ridge garante que o sistema sempre tenha solucao unica, mesmo com rank deficiente

In [ ]:
# === Rank: Visualizacao ===

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Caso 1: Rank completo (3 vetores independentes em 3D)
from mpl_toolkits.mplot3d import Axes3D

A_full = np.array([[1, 2, 0], [0, 1, 2], [2, 1, 1]], dtype=float)
A_rank1 = np.array([[1, 2, 3], [2, 4, 6], [3, 6, 9]], dtype=float)
A_rank2 = np.array([[1, 0, 1], [0, 1, 1], [1, 1, 2]], dtype=float)

cases = [
    ('Rank 3 (completo)\n3 direcoes independentes', A_full, 'green'),
    ('Rank 2\nLinhas vivem em um plano', A_rank2, 'orange'),
    ('Rank 1\nTodas as linhas colineares', A_rank1, 'red'),
]

for ax, (title, M, color) in zip(axes, cases):
    rank = np.linalg.matrix_rank(M)
    svd_vals = np.linalg.svd(M, compute_uv=False)

    # Mostrar a matriz como heatmap
    im = ax.imshow(M, cmap='Blues', aspect='equal')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f'{M[i,j]:.0f}', ha='center', va='center',
                    fontsize=14, fontweight='bold')
    ax.set_xticks(range(M.shape[1]))
    ax.set_yticks(range(M.shape[0]))
    ax.set_xticklabels([f'c{j}' for j in range(M.shape[1])])
    ax.set_yticklabels([f'r{i}' for i in range(M.shape[0])])

    # Borda colorida pelo rank
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(3)

    ax.set_title(f'{title}\nrank={rank}, det={np.linalg.det(M):.1f}\nσ={svd_vals.round(1)}',
                 fontsize=10, fontweight='bold', color=color)

fig.suptitle('Rank e Independencia Linear', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Verificacoes numericas
print("=== Verificacoes ===")
print(f"A_full: rank={np.linalg.matrix_rank(A_full)}, det={np.linalg.det(A_full):.2f}")
print(f"A_rank2: rank={np.linalg.matrix_rank(A_rank2)}, det={np.linalg.det(A_rank2):.2f}")
print(f"  Nota: linha 2 = linha 0 + linha 1 ({A_rank2[2]} = {A_rank2[0]} + {A_rank2[1]})")
print(f"A_rank1: rank={np.linalg.matrix_rank(A_rank1)}, det={np.linalg.det(A_rank1):.2f}")
print(f"  Nota: linha 1 = 2*linha 0, linha 2 = 3*linha 0")

### O que concluir

- Autovalores medem a 'força' de uma transformação em cada direção
- Autovetores definem as direções principais de transformação
- Decomposição em autovalores revela estrutura da matriz

### O que observar

- Estrutura da matriz após operação elementar
- Efeito das operações de linha na estrutura geral
- Quando a matriz fica singular (determinante zero)?

## 9. Inversa e Pseudo-inversa

**Intuicao:** A inversa de uma matriz "desfaz" a transformacao. Se A roda 30 graus, A^(-1) roda -30 graus. Se A escala 2x, A^(-1) escala 0.5x.

**Quando existe A^(-1):**
- A deve ser quadrada (m = n)
- det(A) != 0 (rank completo)
- Resultado: A @ A^(-1) = A^(-1) @ A = I

**Pseudo-inversa A^+ (Moore-Penrose):**
Generaliza para matrizes retangulares ou singulares. Sempre existe!
- Se A e (m x n) com m > n (sobredeterminado): A^+ @ b = solucao de minimos quadrados
- Se A e (m x n) com m < n (subdeterminado): A^+ @ b = solucao de norma minima

**Quando usar cada um em ML:**

| Situacao | Funcao NumPy | Quando |
|----------|-------------|--------|
| A quadrada invertivel | `np.linalg.solve(A, b)` | Mais rapido e estavel que inv |
| A retangular | `np.linalg.lstsq(A, b)` | Regressao linear (caso geral) |
| Precisa de A^(-1) explicitamente | `np.linalg.inv(A)` | Raro -- evite se possivel |
| A pode ser singular | `np.linalg.pinv(A)` | Pseudo-inversa (sempre funciona) |

**Regra de ouro:** Nunca calcule A^(-1) @ b quando pode usar solve(A, b). Inverter e mais lento e menos estavel numericamente.

## 10. Autovalores e Autovetores: Av = lambda * v

**Intuicao:** Um autovetor e uma direcao "especial" de uma transformacao -- a unica direcao que NAO muda de direcao, apenas estica ou comprime. O autovalor diz quanto estica (lambda > 1) ou comprime (0 < lambda < 1).

**Analogia:** Imagine uma porta girando em torno de suas dobradicas. A maioria dos pontos da porta muda de posicao E direcao. Mas os pontos exatamente nas dobradicas so se movem na propria direcao das dobradicas (autovetor) -- a quantidade de movimento e o autovalor.

**Definicao:** A @ v = lambda * v, onde v != 0

**Por que importa em ML:**
- **PCA:** Os autovetores da matriz de covariancia sao as direcoes de maxima variancia. Os autovalores dizem quanta variancia cada direcao explica
- **Estabilidade:** Em sistemas dinamicos e redes recorrentes, |lambda| > 1 causa explosao (exploding gradients), |lambda| < 1 causa desaparecimento (vanishing gradients)
- **Espectral clustering:** Autovetores da matriz Laplaciana do grafo revelam clusters

In [ ]:
# === Autovalores e Autovetores: Visualizacao ===

A = np.array([[3, 1], [1, 3]], dtype=float)
eigenvals, eigenvecs = np.linalg.eig(A)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Painel 1: Autovetores como direcoes especiais ---
ax = axes[0]

# Circulo unitario e transformado
theta = np.linspace(0, 2*np.pi, 200)
circle = np.array([np.cos(theta), np.sin(theta)])
ellipse = A @ circle

ax.plot(circle[0], circle[1], 'gray', alpha=0.4, linewidth=1, label='Circulo unitario')
ax.plot(ellipse[0], ellipse[1], 'steelblue', linewidth=2, label='Transformado por A')

# Autovetores (antes e depois da transformacao)
colors = ['red', 'darkgreen']
for i in range(2):
    v = eigenvecs[:, i]
    lam = eigenvals[i]

    # Autovetor original (normalizado)
    ax.annotate('', xy=v*1.5, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=colors[i], lw=2.5))
    # Autovetor transformado (A @ v = lambda * v)
    ax.annotate('', xy=lam*v*1.5, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=colors[i], lw=2.5, linestyle='--'))

    ax.annotate(f'v{i+1} (λ={lam:.1f})', xy=v*1.7, fontsize=10,
                fontweight='bold', color=colors[i])

ax.set_title('Autovetores: direcoes que NAO mudam\n(solido=original, tracejado=transformado)',
             fontsize=11, fontweight='bold')
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)

# --- Painel 2: Verificacao Av = lambda*v ---
ax = axes[1]

for i in range(2):
    v = eigenvecs[:, i]
    lam = eigenvals[i]
    Av = A @ v

    x_pos = [0, 1, 2]
    labels = [f'v{i+1}', f'A@v{i+1}', f'λ{i+1}*v{i+1}']
    values_x = [v[0], Av[0], lam*v[0]]
    values_y = [v[1], Av[1], lam*v[1]]

    y_offset = i * 0.3
    ax.barh([f'{labels[0]} [x]', f'{labels[0]} [y]'],
            [v[0], v[1]], height=0.3, alpha=0.6, color=colors[i], label=f'Autovetor {i+1}')

ax.set_title('Verificacao: A @ v = λ * v', fontsize=11, fontweight='bold')

# Tabela de verificacao
ax.axis('off')
table_data = []
for i in range(2):
    v = eigenvecs[:, i]
    lam = eigenvals[i]
    Av = A @ v
    table_data.append([f'v{i+1}', f'({v[0]:.3f}, {v[1]:.3f})',
                       f'({Av[0]:.3f}, {Av[1]:.3f})',
                       f'({lam*v[0]:.3f}, {lam*v[1]:.3f})',
                       f'{lam:.1f}',
                       f'{np.allclose(Av, lam*v)}'])

table = ax.table(cellText=table_data,
                 colLabels=['', 'v', 'A@v', 'λ*v', 'λ', 'Iguais?'],
                 cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 2.0)

plt.tight_layout()
plt.show()

print(f"Matriz A =\n{A}")
print(f"\nAutovalores: λ1={eigenvals[0]:.1f}, λ2={eigenvals[1]:.1f}")
print(f"Autovetores: v1={eigenvecs[:,0].round(3)}, v2={eigenvecs[:,1].round(3)}")
print(f"\nNota: A e simetrica, entao autovetores sao ortogonais:")
print(f"  v1 . v2 = {eigenvecs[:,0] @ eigenvecs[:,1]:.6f} (≈ 0)")

### O que observar

- Estrutura da matriz após operação elementar
- Efeito das operações de linha na estrutura geral
- Quando a matriz fica singular (determinante zero)?

## 11. PCA do Zero com Autovalores

**Intuicao:** PCA encontra as "melhores perspectivas" para olhar seus dados. Imagine fotografar uma escultura: de certos angulos voce ve mais detalhes que de outros. PCA encontra o angulo que mostra a MAXIMA variacao nos dados.

**Algoritmo passo a passo:**
1. Centralizar dados: X_c = X - media(X)
2. Calcular matriz de covariancia: C = X_c^T @ X_c / (n-1)
3. Calcular autovalores/autovetores de C
4. Ordenar por autovalor decrescente (maior variancia primeiro)
5. Projetar: X_pca = X_c @ V (V = autovetores escolhidos)

**Variancia explicada:** lambda_i / soma(lambdas) = % da variancia capturada pela i-esima componente

**Por que funciona:** A matriz de covariancia e simetrica positivo-semi-definida, entao seus autovetores sao ortogonais -- formam um novo sistema de coordenadas alinhado com as direcoes de maxima variacao.

#### por que em ml

- PCA: reduz dimensionalidade achando autovetores da covariância
- Essencial para visualização de dados altos-dimensionais
- Preprocessing padrão em muitos pipelines

In [ ]:
# === PCA do Zero: Implementacao Didatica com Visualizacao ===

# 1. Gerar dados correlacionados
np.random.seed(42)
n = 150
mean = [2, 3]
cov_true = [[3.0, 2.5], [2.5, 3.0]]  # Dados correlacionados
data = np.random.multivariate_normal(mean, cov_true, n)

# 2. Centralizar
data_centered = data - data.mean(axis=0)
print("Passo 1: Centralizar (subtrair media)")
print(f"  Media original: {data.mean(axis=0).round(3)}")
print(f"  Media apos centralizar: {data_centered.mean(axis=0).round(6)}")

# 3. Covariancia
cov_matrix = np.cov(data_centered.T)
print(f"\nPasso 2: Matriz de Covariancia")
print(f"  {cov_matrix.round(3)}")

# 4. Autovalores/autovetores
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
# Ordenar decrescente
idx = eigenvalues.argsort()[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]
print(f"\nPasso 3: Autovalores (variancia por direcao)")
print(f"  λ1 = {eigenvalues[0]:.3f} ({eigenvalues[0]/eigenvalues.sum()*100:.1f}% da variancia)")
print(f"  λ2 = {eigenvalues[1]:.3f} ({eigenvalues[1]/eigenvalues.sum()*100:.1f}% da variancia)")

# 5. Projetar
data_pca = data_centered @ eigenvectors

# === Visualizacao: 3 Paineis ===
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Painel 1: Dados originais com componentes principais
ax = axes[0]
ax.scatter(data_centered[:, 0], data_centered[:, 1], c='steelblue', s=15, alpha=0.5)
for i, (val, vec, color, name) in enumerate(
    zip(eigenvalues, eigenvectors.T, ['red', 'orange'], ['PC1', 'PC2'])):
    scale = np.sqrt(val) * 2
    ax.annotate('', xy=vec * scale, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=3))
    ax.annotate(f'{name}\nλ={val:.2f} ({val/eigenvalues.sum()*100:.0f}%)',
                xy=vec * scale * 1.15, fontsize=9, fontweight='bold', color=color)
ax.set_title('Dados + Componentes Principais', fontsize=11, fontweight='bold')
ax.set_xlabel('x1 (centralizado)')
ax.set_ylabel('x2 (centralizado)')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Painel 2: Projecao em PC1 (reducao para 1D)
ax = axes[1]
proj_1d = data_centered @ eigenvectors[:, 0:1] @ eigenvectors[:, 0:1].T
ax.scatter(data_centered[:, 0], data_centered[:, 1], c='lightgray', s=15, alpha=0.4, label='Original')
ax.scatter(proj_1d[:, 0], proj_1d[:, 1], c='red', s=15, alpha=0.5, label='Projecao em PC1')
# Linhas de erro de reconstrucao
for j in range(0, n, 5):
    ax.plot([data_centered[j, 0], proj_1d[j, 0]],
            [data_centered[j, 1], proj_1d[j, 1]], 'gray', alpha=0.2, linewidth=0.5)
ax.set_title('Projecao em PC1 (2D -> 1D)', fontsize=11, fontweight='bold')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_aspect('equal')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Painel 3: Dados no espaco PCA
ax = axes[2]
ax.scatter(data_pca[:, 0], data_pca[:, 1], c='steelblue', s=15, alpha=0.5)
ax.set_xlabel(f'PC1 ({eigenvalues[0]/eigenvalues.sum()*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({eigenvalues[1]/eigenvalues.sum()*100:.1f}% var)')
ax.set_title('Dados no Espaco PCA\n(componentes descorrelacionadas)', fontsize=11, fontweight='bold')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Verificar descorrelacao
corr_pca = np.corrcoef(data_pca.T)[0, 1]
ax.annotate(f'Correlacao PC1-PC2: {corr_pca:.6f}\n(≈ 0, como esperado!)',
            xy=(0.05, 0.95), xycoords='axes fraction', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.tight_layout()
plt.show()

# Verificacao com sklearn
# sklearn verification disabled
# data_sk = pca_sk.fit_transform(data_centered)
# print(f"\n=== Verificacao com sklearn ===")
print(f"Variancia explicada (nosso): {(eigenvalues/eigenvalues.sum()*100).round(1)}")
# # print(f"Variancia explicada (sklearn): {(pca_sk.explained_variance_ratio_*100).round(1)}")
# # print(f"Resultados iguais? {np.allclose(abs(data_pca), abs(data_sk))}")

#### conexao com

- **Regressão Linear (1.2)**: Matriz X de features, vetor y de targets
- **Solução**: (X^T X)^-1 X^T y envolve inversa de matriz

### Erro

**Esquecer que @ é não-comutativa (A@B ≠ B@A):**

```python# ❌ ERRADO: pensar que ordem não importaresultado = B @ A# ✓ CORRETO: verificar dimensõesA: (m, n), B: (n, p) → A@B: (m, p)B: (n, p), A: (m, n) → B@A: NÃO COMPATÍVEL se p ≠ m```

#### conexao com

- **Regressão Linear (1.2)**: Matriz X de features, vetor y de targets
- **Solução**: (X^T X)^-1 X^T y envolve inversa de matriz

**O Que Observar e Concluir**

**Painel Esquerdo (Dados + Componentes):**
- PC1 (vermelho) aponta na direcao de MAIOR espalhamento dos dados
- PC2 (laranja) e perpendicular a PC1 (autovetores sao ortogonais para matrizes simetricas)
- O comprimento das setas e proporcional ao autovalor (variancia naquela direcao)

**Painel Central (Projecao em PC1):**
- Os pontos cinzas (originais 2D) sao projetados nos pontos vermelhos (sobre PC1)
- As linhas cinzas sao o "erro de reconstrucao" -- quanto perdemos ao descartar PC2
- A maioria das linhas e curta: pouca informacao perdida

**Painel Direito (Espaco PCA):**
- No novo sistema de coordenadas, PC1 e PC2 sao DESCORRELACIONADAS (correlacao ≈ 0)
- A variancia total e preservada, apenas redistribuida entre componentes independentes
- Se mantivermos apenas PC1, capturamos a maior parte da variancia com metade das dimensoes

**Conexao fundamental:** PCA e uma mudanca de base que alinha os eixos com as direcoes de maxima variancia. Isso remove correlacoes e permite compressao.

## 12. SVD: A = U . Sigma . V^T

**Intuicao:** SVD e como decompor qualquer transformacao em 3 passos simples: rotacao (V^T), escala (Sigma), rotacao (U). Funciona para QUALQUER matriz (nao precisa ser quadrada nem simetrica).

**Componentes:**
- **V^T** (n x n): Rotacao no espaco de entrada
- **Sigma** (m x n): Escala (valores singulares na diagonal). sigma_1 >= sigma_2 >= ... >= 0
- **U** (m x m): Rotacao no espaco de saida

**Relacao com autovalores:** sigma_i = sqrt(lambda_i) dos autovalores de A^T @ A

**Aplicacoes em ML:**
- **PCA:** Pode ser calculado via SVD (mais estavel numericamente que eig da covariancia)
- **Compressao:** Manter apenas os k maiores valores singulares (rank-k approximation)
- **Recomendacao:** Fatorar a matriz user-item em fatores latentes
- **Pseudo-inversa:** A^+ = V @ Sigma^+ @ U^T (construida diretamente da SVD)
- **Regularizacao:** Valores singulares pequenos indicam direcoes sensiveis a ruido

In [ ]:
# === SVD Basico: Decomposicao e Reconstrucao ===

A_svd = np.array([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9],
                   [10, 11, 12]], dtype=float)

U, s, Vt = np.linalg.svd(A_svd, full_matrices=False)

print(f"Matriz A: {A_svd.shape}")
print(f"U: {U.shape}  (rotacao no espaco de saida)")
print(f"s: {s.shape}  (valores singulares)")
print(f"Vt: {Vt.shape} (rotacao no espaco de entrada)")
print(f"\nValores singulares: {s.round(3)}")
print(f"  σ1={s[0]:.3f} >> σ2={s[1]:.3f} >> σ3={s[2]:.6f} ≈ 0")
print(f"  Rank efetivo: {np.sum(s > 1e-10)} (σ3 ≈ 0 porque linhas sao quase colineares)")

# Reconstrucao: A = U @ diag(s) @ Vt
A_reconstruida = U @ np.diag(s) @ Vt
print(f"\n=== Verificacao: A = U @ Sigma @ Vt ===")
print(f"Erro de reconstrucao: {np.linalg.norm(A_svd - A_reconstruida):.2e}")
print(f"Reconstrucao perfeita? {np.allclose(A_svd, A_reconstruida)}")

# Propriedades de U e Vt
print(f"\n=== Propriedades ===")
print(f"U^T @ U ≈ I? {np.allclose(U.T @ U, np.eye(U.shape[1]))}")
print(f"Vt @ Vt^T ≈ I? {np.allclose(Vt @ Vt.T, np.eye(Vt.shape[0]))}")

# Relacao com autovalores de A^T @ A
eigvals_ATA = np.linalg.eigvalsh(A_svd.T @ A_svd)[::-1]
print(f"\nRelacao sigma_i = sqrt(lambda_i de A^T @ A):")
print(f"  Valores singulares:  {s.round(3)}")
print(f"  sqrt(autovalores):   {np.sqrt(np.maximum(eigvals_ATA, 0)).round(3)}")
print(f"  Iguais? {np.allclose(s, np.sqrt(np.maximum(eigvals_ATA, 0)))}")

### Erro

**Esquecer que @ é não-comutativa (A@B ≠ B@A):**

```python# ❌ ERRADO: pensar que ordem não importaresultado = B @ A# ✓ CORRETO: verificar dimensõesA: (m, n), B: (n, p) → A@B: (m, p)B: (n, p), A: (m, n) → B@A: NÃO COMPATÍVEL se p ≠ m```

### O que concluir

- Rank determina dimensionalidade efetiva dos dados
- Matrizes de rank completo são invertíveis
- Rank reduzido indica redundância ou colinearidade

### O que observar

- Valores da diagonal principal em matrizes diagonalizadas
- Como os autovetores se relacionam geometricamente
- Interpretação das direções principais

#### por que em ml

- SVD: mais estável numericamente que autovalores
- Usado em recomendação, compressão e denoising
- Generaliza para matrizes não-quadradas (dados reais)

## 13. Compressao de Imagem com SVD

**Intuicao:** Cada valor singular captura um "nivel de detalhe" da imagem. Os maiores capturam estrutura geral (contornos), os menores capturam ruido e detalhes finos. Manter apenas os k maiores e como olhar a imagem de longe: voce perde detalhes mas reconhece o que e.

**Aproximacao rank-k:**
A_k = U[:, :k] @ diag(sigma[:k]) @ V^T[:k, :]

**Custo de armazenamento:**
- Original: m * n valores
- Rank-k: k * (m + n + 1) valores
- Compressao: (m * n) / (k * (m + n + 1))

In [ ]:
# === Compressao de Imagem com SVD ===

# Create synthetic 8x8 digit image instead of load_digits()
img = np.array([[0, 0, 1, 1, 1, 1, 0, 0],
                 [0, 1, 1, 1, 1, 1, 1, 0],
                 [1, 1, 0, 0, 0, 0, 1, 1],
                 [1, 1, 0, 0, 0, 0, 1, 1],
                 [0, 1, 1, 0, 0, 1, 1, 0],
                 [0, 0, 1, 1, 1, 1, 0, 0],
                 [0, 0, 0, 1, 1, 0, 0, 0],
                 [0, 0, 0, 0, 0, 0, 0, 0]], dtype=float)
m, n = img.shape

U, s, Vt = np.linalg.svd(img, full_matrices=False)

# Visualizar compressao em diferentes ranks
ranks = [1, 2, 3, 4, 6, 8]
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for i, r in enumerate(ranks):
    # Reconstruir com rank r
    U_r = U[:, :r]
    s_r = s[:r]
    Vt_r = Vt[:r, :]
    img_approx = U_r @ np.diag(s_r) @ Vt_r

    ax = axes[i]
    im = ax.imshow(img_approx, cmap='gray')

    # Calcular erro
    err = np.linalg.norm(img - img_approx, 'fro')

    ax.set_title(f'Rank {r}, Erro: {err:.2f}')
    ax.axis('off')

# Hide the last two subplots
axes[-2].axis('off')
axes[-1].axis('off')

plt.suptitle('Compressao com SVD: Reduzindo para Diferentes Ranks', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Imagem original: {img.shape} (8x8 = 64 elementos)")
print(f"Rank-1: U[0:8,0:1] + s[0:1] + Vt[0:1, 0:8] = 8+1+8 = 17 parametros (73% compressao)")
print(f"Rank-2: 8+2+8 = 18 parametros (72% compressao)")
print(f"Rank-4: 8+4+8 = 20 parametros (69% compressao)")

### O que observar

- Valores da diagonal principal em matrizes diagonalizadas
- Como os autovetores se relacionam geometricamente
- Interpretação das direções principais

### O que concluir

- Rank determina dimensionalidade efetiva dos dados
- Matrizes de rank completo são invertíveis
- Rank reduzido indica redundância ou colinearidade

#### por que em ml

- Regressão linear: solução analítica é (X^T X)^-1 X^T y
- Entender inversão de matrizes é essencial para entender regressão

## 14. Recomendacao com SVD (Collaborative Filtering)

**Intuicao:** Se voce tem uma matriz de avaliacoes users x filmes (com muitos valores faltantes), SVD encontra "gostos latentes". Por exemplo, um fator latente pode representar "gosta de acao" -- usuarios com peso alto nesse fator tendem a gostar de filmes com peso alto no mesmo fator.

**Como funciona:**
1. Preencher valores faltantes (media simples ou mais sofisticado)
2. Decompor: R ≈ U_k @ Sigma_k @ V_k^T (rank-k reduzido)
3. Cada usuario vira um vetor de k dimensoes (linha de U_k @ Sigma_k)
4. Cada item vira um vetor de k dimensoes (coluna de Sigma_k @ V_k^T)
5. Predicao: score(user, item) = vetor_usuario . vetor_item

**Por que rank baixo funciona:** Se existem apenas k "gostos" fundamentais (acao, romance, comedia...), a matriz de avaliacoes verdadeira tem rank ≈ k, mesmo que tenha milhoes de usuarios e milhares de filmes.

In [ ]:
# === Recomendacao com SVD ===

# Matriz de avaliacoes: 5 usuarios x 5 filmes (0 = nao avaliado)
ratings = np.array([
    [5, 4, 0, 1, 0],   # Usuario 0: gosta de acao (filmes 0,1)
    [4, 5, 0, 1, 0],   # Usuario 1: gosta de acao
    [0, 0, 5, 4, 5],   # Usuario 2: gosta de drama (filmes 2,3,4)
    [0, 0, 5, 4, 4],   # Usuario 3: gosta de drama
    [3, 3, 2, 0, 1]    # Usuario 4: misto
], dtype=float)

# Passo 1: Preencher valores faltantes com media
ratings_filled = ratings.copy()
mask = ratings == 0
mean_val = ratings[~mask].mean()
ratings_filled[mask] = mean_val
print(f"Media global: {mean_val:.2f}")
print(f"Matriz preenchida:\n{ratings_filled.round(1)}")

# Passo 2: SVD com rank k=2 (2 "gostos latentes")
U, s, Vt = np.linalg.svd(ratings_filled, full_matrices=False)
rank = 2
ratings_pred = (U[:, :rank] @ np.diag(s[:rank]) @ Vt[:rank, :]).clip(0, 5)

print(f"\nValores singulares: {s.round(2)}")
print(f"Variancia capturada por rank-2: {(s[:rank]**2).sum() / (s**2).sum() * 100:.1f}%")

# Passo 3: Previsoes
print(f"\n=== Previsoes (rank-{rank}) ===")
print(f"Predicoes:\n{ratings_pred.round(1)}")

# Identificar recomendacoes (onde rating original era 0)
print(f"\n=== Recomendacoes (itens nao avaliados) ===")
for user in range(ratings.shape[0]):
    not_rated = np.where(ratings[user] == 0)[0]
    if len(not_rated) > 0:
        scores = [(item, ratings_pred[user, item]) for item in not_rated]
        scores.sort(key=lambda x: -x[1])
        best = scores[0]
        print(f"  Usuario {user}: recomendar Filme {best[0]} (score previsto: {best[1]:.1f})")

## 15. Matrizes em ML Moderno: Mapa de Aplicacoes

Quase toda operacao em ML se resume a uma operacao matricial. Aqui esta o mapa completo:

| Operacao | Formula | Onde Aparece |
|----------|---------|-------------|
| **Forward pass** | Y = X @ W + b | Toda rede neural, toda camada |
| **Normal equation** | beta = (X^T X)^(-1) X^T y | Regressao linear (solucao fechada) |
| **Kernel matrix** | K_ij = k(x_i, x_j) | SVM, Gaussian Processes |
| **Embedding lookup** | e = E[id, :] | NLP, sistemas de recomendacao |
| **Attention** | A = softmax(Q K^T / sqrt(d)) V | Transformers (GPT, BERT, etc.) |
| **Batch Norm** | x_hat = (x - mu) / sigma | Normalizacao em redes profundas |
| **Convolution** | Y = im2col(X) @ W_flat | CNNs (conv como multiplicacao) |
| **Gram matrix** | G = F^T @ F | Neural Style Transfer |

**Insight:** Entender operacoes matriciais e entender ML. Otimizar operacoes matriciais (GPUs, TPUs) e o que torna deep learning viavel.

## 16. Exercicios Praticos <a id='16-exercicios'></a>

### Exercicio 1 — Transformacao de Pontos com Rotacao

**Instrucoes:**
Dado uma matriz A de rotacao, transforme um conjunto de 5 pontos aleatorios.

Tarefas:
1. Definir a matriz A = [[2, 1], [1, 2]]
2. Gerar 5 pontos aleatorios P (2x5)
3. Transformar: P_trans = A @ P
4. Calcular det(A) e interpretar o resultado
5. Visualizar os pontos originais e transformados

**Dica:** np.random.rand, np.linalg.det, plt.scatter

In [ ]:
# === Exercicio 1 — Pratica ===
# Implemente aqui:

# 1. Definir a matriz A = [[2, 1], [1, 2]]
A_ex = None  # TAREFA DO ALUNO

# 2. Gerar 5 pontos aleatorios P (2x5) no intervalo [-2, 2]
P = None  # TAREFA DO ALUNO

# 3. Calcular det(A)
det_A = None  # TAREFA DO ALUNO

# 4. Transformar: P_trans = A @ P
P_trans = None  # TAREFA DO ALUNO

# 5. Exibir resultados
# print(f"det(A) = {det_A}")
# print(f"Pontos originais:\n{P}")
# print(f"Pontos transformados:\n{P_trans}")

# 6. (Bonus) Visualizar com plt.scatter
# plt.scatter(P[0], P[1], label='Original')
# plt.scatter(P_trans[0], P_trans[1], label='Transformado')
# plt.legend()
# plt.show()

### O que observar

- Magnitude dos valores singulares em SVD- Quantos valores singulares são significativos?

- Taxa de decaimento na compressão

#### por que em ml

- Rank da matriz determina graus de liberdade efetivos
- Rank deficiente indica overfitting ou colinearidade de features
- Regularização aumenta condicionamento (reduz rank efetivo)

In [ ]:
# === Exercicio 1 — Solucao ===

A_ex = np.array([[2, 1], [1, 2]], dtype=float)
P = np.random.rand(2, 5) * 4 - 2  # 5 pontos aleatorios em [-2, 2]
P_trans = A_ex @ P
det_A = np.linalg.det(A_ex)

print('Matriz A:')
print(A_ex)
print(f'\ndet(A) = {det_A:.2f}')
print(f'Interpretacao: areas expandidas por fator {det_A:.1f}, orientacao preservada (det > 0)')
print(f'\nPontos originais (2x5):\n{P.round(2)}')
print(f'\nPontos transformados (2x5):\n{P_trans.round(2)}')

# Visualizacao
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(P[0], P[1], s=100, c='blue', marker='o', label='Original', zorder=5)
ax.scatter(P_trans[0], P_trans[1], s=100, c='red', marker='^', label='Transformado', zorder=5)
for i in range(5):
    ax.annotate('', xy=(P_trans[0, i], P_trans[1, i]), xytext=(P[0, i], P[1, i]),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1))
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title(f'Transformacao por A (det={det_A:.1f})', fontweight='bold')
plt.show()

#### conexao com

- **Bases de Dados**: Matriz de features de usuários
- **Índices**: Estrutura matricial de índices para busca rápida
- **Queries**: JOIN em SQL é operação matricial implícita

### O que observar

- Magnitude dos valores singulares em SVD- Quantos valores singulares são significativos?

- Taxa de decaimento na compressão

#### conexao com

- **Redes Neurais (2.1)**: Cada camada é multiplicação matricial
- **Pesos**: Matriz W, entrada x, output Wx + b

### Erro

**Não verificar shape antes de operações matriciais:**

```python# ❌ ERRADO: assume que shapes são compatíveisC = A @ B# ✓ CORRETO: verificar sempreif A.shape[1] != B.shape[0]:    raise ValueError(f"Incompatível: {A.shape} @ {B.shape}")C = A @ B```

#### conexao com

- **Redes Neurais (2.1)**: Cada camada é multiplicação matricial
- **Pesos**: Matriz W, entrada x, output Wx + b

### O que concluir

- Determinante é zero quando a matriz é singular (não-invertível)- Determinante mede volume de transformação
- Sinal do determinante indica orientação da transformação

### Exercicio 2 -- Regressao Polinomial com Minimos Quadrados

**Instrucoes:**
Dado dados ruidosos gerados por y = 2x^2 - 3x + 1, recupere os coeficientes usando minimos quadrados.

Tarefas:
1. Gerar x e y com ruido
2. Construir a matriz de design X = [1, x, x^2] (cada coluna e uma feature)
3. Resolver beta = lstsq(X, y) para encontrar os coeficientes
4. Comparar coeficientes encontrados com os verdadeiros [1, -3, 2]
5. Visualizar dados e curva ajustada

**Dica:** np.column_stack, np.linalg.lstsq

In [ ]:
# === Exercicio 2 — Pratica ===

# Dados (ja fornecidos)
x_ex = np.linspace(-2, 2, 10)
y_ex = 2*x_ex**2 - 3*x_ex + 1 + 0.5*np.random.randn(10)

# TAREFA DO ALUNO: Construir matriz de design X (10 x 3)
# X_design = np.column_stack([...])

# TAREFA DO ALUNO: Resolver por minimos quadrados
# beta = np.linalg.lstsq(..., rcond=None)[0]

# TAREFA DO ALUNO: Exibir coeficientes
# print(f"Coeficientes encontrados: {beta}")
# print(f"Coeficientes verdadeiros: [1, -3, 2]")

### O que concluir

- Determinante é zero quando a matriz é singular (não-invertível)- Determinante mede volume de transformação
- Sinal do determinante indica orientação da transformação

In [ ]:
# === Exercicio 2 — Solucao ===

x_ex = np.linspace(-2, 2, 10)
y_ex = 2*x_ex**2 - 3*x_ex + 1 + 0.5*np.random.randn(10)

# Matriz de design: cada coluna e uma "feature" do polinomio
X_design = np.column_stack([np.ones(10), x_ex, x_ex**2])
print(f"Matriz de design X ({X_design.shape}):")
print(f"  Colunas: [1, x, x^2]")

# Minimos quadrados: beta = argmin ||X @ beta - y||^2
beta, residuals, rank, sv = np.linalg.lstsq(X_design, y_ex, rcond=None)

print(f'\nCoeficientes encontrados: intercept={beta[0]:.2f}, x={beta[1]:.2f}, x^2={beta[2]:.2f}')
print(f'Coeficientes verdadeiros: intercept=1, x=-3, x^2=2')

# Visualizacao
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_ex, y_ex, s=60, c='steelblue', zorder=5, label='Dados (com ruido)')
x_smooth = np.linspace(-2, 2, 100)
y_smooth = beta[0] + beta[1]*x_smooth + beta[2]*x_smooth**2
y_true = 1 - 3*x_smooth + 2*x_smooth**2
ax.plot(x_smooth, y_smooth, 'r-', linewidth=2, label=f'Ajuste: {beta[2]:.1f}x² + ({beta[1]:.1f})x + {beta[0]:.1f}')
ax.plot(x_smooth, y_true, 'g--', linewidth=1.5, alpha=0.7, label='Verdadeiro: 2x² - 3x + 1')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Regressao Polinomial via Minimos Quadrados (Algebra Linear!)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

# R² score
y_pred = X_design @ beta
ss_res = np.sum((y_ex - y_pred)**2)
ss_tot = np.sum((y_ex - y_ex.mean())**2)
r2 = 1 - ss_res / ss_tot
print(f'\nR² = {r2:.4f}')
print(f'Nota: regressao polinomial e apenas um sistema linear Ax=b disfarçado!')

### O que observar

- Erro de reconstrução com SVD truncado
- Qualidade visual da imagem comprimida
- Trade-off entre compressão e qualidade

### Erro

**Não verificar shape antes de operações matriciais:**

```python# ❌ ERRADO: assume que shapes são compatíveisC = A @ B# ✓ CORRETO: verificar sempreif A.shape[1] != B.shape[0]:    raise ValueError(f"Incompatível: {A.shape} @ {B.shape}")C = A @ B```

## 17. Erros Comuns e Armadilhas

**1. Confundir multiplicacao elemento-a-elemento com matricial**
- Errado: `A * B` (multiplica elemento por elemento, exige mesmo shape)
- Certo: `A @ B` (multiplicacao matricial, exige compatibilidade de dimensoes)
- NumPy nao avisa quando voce usa `*` em vez de `@` e os shapes coincidem!

**2. Calcular A^(-1) @ b em vez de usar solve**
- Errado: `x = np.linalg.inv(A) @ b` (lento, instavel numericamente)
- Certo: `x = np.linalg.solve(A, b)` (2x mais rapido, mais preciso)
- Regra: so calcule a inversa se REALMENTE precisa de A^(-1) como matriz

**3. Esquecer de centralizar dados antes de PCA**
- Sem centralizar, PC1 aponta para a MEDIA dos dados, nao para a direcao de maxima variancia
- Sempre: `X_centered = X - X.mean(axis=0)` antes de calcular covariancia
- sklearn.PCA faz isso automaticamente, mas se implementar do zero, nao esqueca

**4. Assumir que autovalores sao sempre reais**
- Para matrizes simetricas (covariancia, kernel): sim, sempre reais
- Para matrizes gerais: podem ser complexos!
- Use `np.linalg.eigh` para simetricas (garante reais e ordenados), `np.linalg.eig` para gerais

**5. Overflow em SVD com matrizes grandes**
- Problema: `np.linalg.svd(A)` com full_matrices=True cria matrizes enormes
- Solucao: usar `full_matrices=False` (SVD "economica")
- Melhor ainda: `scipy.sparse.linalg.svds(A, k=10)` para apenas os k maiores valores singulares

**6. Confundir rank da matriz com numero de linhas/colunas**
- Uma matriz 1000x1000 pode ter rank = 1 (todas as linhas sao multiplos uma da outra)
- Rank = numero de direcoes INDEPENDENTES, nao o tamanho da matriz
- Em dados reais, rank efetivo e frequentemente muito menor que min(m, n)

**7. Broadcasting silencioso causando bugs**
- `X + b` onde X e (100, 5) e b e (5,) funciona: b e somado a cada linha
- `X + b` onde X e (100, 5) e b e (100,) FALHA ou faz broadcasting inesperado
- Sempre verifique shapes com `.shape` antes de operacoes entre matrizes de tamanhos diferentes

#### por que em ml

- Determinante relacionado a volume transformado por rede neural
- Jacobiano (matriz de derivadas) determina sensibilidade do modelo
- Hessiano (matriz de segundas derivadas) usado em otimização avançada

## 18. Resumo Final

| Conceito | Formula / Operacao | Aplicacao em ML |
|----------|-------------------|----------------|
| Tipos de matrizes | I, D, S=S^T, Q^T Q=I | Inicializacao, covariancia, rotacao |
| Multiplicacao | (A@B)_ij = sum_k a_ik b_kj | Forward pass: Y = X @ W + b |
| Transformacoes 2D | Colunas de A = imagem dos vetores base | Camadas lineares de redes neurais |
| Sistemas lineares | Ax = b | Regressao: X @ beta = y |
| Determinante | det(A) = fator de escala de area/volume | Invertibilidade, Jacobiano, Normalizing Flows |
| Rank | num. direcoes independentes | Multicolinearidade, compressao, PCA |
| Inversa / Pseudo-inv | A^(-1), A^+ | solve(A,b) >> inv(A)@b |
| Autovalores | A @ v = lambda * v | PCA, estabilidade, spectral clustering |
| PCA | Autovetores da covariancia | Reducao de dimensionalidade |
| SVD | A = U @ Sigma @ V^T | Compressao, recomendacao, pseudo-inversa |

### Conexoes com Proximos Topicos

- **Calculo (0.4):** Gradiente e Hessiano sao matrizes. Otimizacao usa multiplicacao matricial
- **Integrais e Series (0.5):** Series de Taylor para aproximar funcoes matriciais (exp(A))
- **Probabilidade (0.6):** Matrizes de covariancia sao simetricas positivo-definidas
- **Regressao Linear (1.2):** beta = (X^T X)^(-1) X^T y (normal equation)
- **Redes Neurais (4.1):** Forward pass = cadeia de multiplicacoes matriciais. Backprop usa Jacobiano
- **Fatoracao de Matrizes:** Recomendacao (SVD), topicos (NMF), compressao

### Checklist de Aprendizado

- [ ] Reconheco os 6 tipos de matrizes e suas propriedades
- [ ] Entendo multiplicacao matricial como encadeamento de transformacoes
- [ ] Sei visualizar e interpretar transformacoes lineares 2D
- [ ] Calculo determinante e interpreto seu significado geometrico
- [ ] Entendo rank e sua relacao com independencia linear
- [ ] Sei quando usar solve vs inv vs pinv vs lstsq
- [ ] Calculo autovalores/autovetores e entendo seu significado
- [ ] Implemento PCA do zero (centralizar -> covariancia -> eig -> projetar)
- [ ] Decomponho matrizes via SVD e uso para compressao
- [ ] Conheco as 7 armadilhas comuns e como evita-las

---
*Parte do Roadmap Completo de ML em Portugues -- 53 notebooks do zero ao avancado.*

In [ ]:
# SOLUCAO - Exercício 1
import numpy as np

# Criar e multiplicar matrizes
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

# Multiplicação matricial
C = A @ B
print("A @ B =")
print(C)
print(f"Shape: {C.shape}")

# Verificar: (2,2) @ (2,2) = (2,2) ✓
assert C.shape == (2, 2)


### O que observar

- Erro de reconstrução com SVD truncado
- Qualidade visual da imagem comprimida
- Trade-off entre compressão e qualidade

#### por que em ml

- Redes neurais: cada camada é y = Wx + b (multiplicação matricial)- Backpropagation: multiplicação por transposta da matriz de pesos
- Toda otimização em deep learning é matricial

In [ ]:
# SOLUCAO - Exercício 2
import numpy as np

# Determinar rank e invertibilidade
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

rank = np.linalg.matrix_rank(A)
det = np.linalg.det(A)

print(f"Rank: {rank}")
print(f"Determinante: {det:.2e}")
print(f"Invertível? {rank == 3}")

# Usar pseudo-inversa (funciona mesmo para singular)
A_pinv = np.linalg.pinv(A)
print(f"Pseudo-inversa shape: {A_pinv.shape}")


In [ ]:
# SOLUCAO - Exercício 3
import numpy as np

# SVD truncado para compressão
A = np.random.randn(10, 8)

# Decompor
U, s, Vt = np.linalg.svd(A, full_matrices=False)

# Truncar para k=3
k = 3
U_k = U[:, :k]
s_k = s[:k]
Vt_k = Vt[:k, :]

# Reconstruir
A_approx = U_k @ np.diag(s_k) @ Vt_k

# Erro
erro = np.linalg.norm(A - A_approx)
print(f"Erro Frobenius: {erro:.6f}")
print(f"Razão compressão: {A.size / (U_k.size + s_k.size + Vt_k.size):.2f}x")


In [ ]:
# EXERCICIO 3: Decomposicao em Autovalores
# TAREFA DO ALUNO: Implemente SVD truncado para reduzir dimensionalidade de uma matriz aleatoria

import numpy as np

# Crie matriz aleatoria 10x8
A = np.random.randn(10, 8)

# TAREFA DO ALUNO: Calcule SVD completo
U, s, Vt = None  # TAREFA DO ALUNO: use np.linalg.svd(A)

# TAREFA DO ALUNO: Mantenha apenas os 3 maiores valores singulares
k = 3
U_k = None  # TAREFA DO ALUNO
s_k = None  # TAREFA DO ALUNO
Vt_k = None  # TAREFA DO ALUNO

# TAREFA DO ALUNO: Reconstrua matriz aproximada
A_approx = None  # TAREFA DO ALUNO: A_k = U_k @ np.diag(s_k) @ Vt_k

# TAREFA DO ALUNO: Calcule erro de reconstrucao
erro_frobenius = None  # TAREFA DO ALUNO: use np.linalg.norm(A - A_approx)

print(f"Forma original: {A.shape}")
print(f"Valores singulares: {s}")
print(f"Erro de reconstrucao (rank-{k}): {erro_frobenius:.6f}")
print(f"Razao de compressao: {8*10 / (10*3 + 3 + 3*8):.2f}x")


In [ ]:
# SOLUCAO Exercicio 3: SVD truncado para reduzir dimensionalidade
import numpy as np

np.random.seed(0)
A = np.random.randn(10, 8)

# SVD economico (full_matrices=False da apenas a porcao util)
U, s, Vt = np.linalg.svd(A, full_matrices=False)

k = 3
U_k = U[:, :k]
s_k = s[:k]
Vt_k = Vt[:k, :]

A_approx = U_k @ np.diag(s_k) @ Vt_k
erro_frobenius = np.linalg.norm(A - A_approx)

print(f"Forma original A: {A.shape}")
print(f"Formas truncadas -> U_k: {U_k.shape}, s_k: {s_k.shape}, Vt_k: {Vt_k.shape}")
print(f"Valores singulares: {np.round(s, 4)}")
print(f"Top-{k} valores singulares: {np.round(s_k, 4)}")
print(f"Erro Frobenius (rank-{k}): {erro_frobenius:.6f}")
print(f"Razao de compressao: {8*10 / (10*k + k + k*8):.2f}x")

### O que concluir

- SVD é mais estável numericamente que decomposição em autovalores
- SVD funciona para matrizes não-quadradas
- Valores singulares ordenados revelam importância dos componentes

#### conexao com

- **PCA (3.1)**: Autovalores/autovetores reduzem dimensionalidade
- **Componentes principais**: Autovetores da matriz de covariância

### Erro

**Confundir rank com shape:**

```python# ❌ ERRADO: achar que matriz 3x3 sempre tem rank 3A = np.array([[1, 2, 3], [2, 4, 6], [3, 6, 9]])rank_A = 3  # FALSO! linhas 2,3 são múltiplos de linha 1# ✓ CORRETO: usar np.linalg.matrix_rankrank_A = np.linalg.matrix_rank(A)  # returns 1```

## Resumo

### Hierarquia de Conceitos

#### Nível 1: Fundamentos

- Tipos de matrizes (zero, identidade, diagonal, etc)- Operações básicas (soma, multiplicação escalar)

#### Nível 2: Operações Principais

- Multiplicação matricial A @ B- Determinante e rank
- Inversa de matrizes

#### Nível 3: Decomposição

- Autovalores e autovetores
- Eigendecomposition (A = QΛQ^-1)- SVD (A = UΣV^T)

#### Nível 4: Aplicações

- PCA para redução de dimensionalidade
- Compressão de imagens
- Recomendação colaborativa

### Checklist de Aprendizado- [ ] Entendo operações matriciais e requisitos de compatibilidade- [ ] Posso calcular multiplicação, determinante e rank manualmente- [ ] Entendo diferença entre autovalores e valores singulares- [ ] Posso aplicar SVD em problemas reais- [ ] Conheço conexões com regressão linear e redes neurais- [ ] Sei usar np.linalg para operações matriciais- [ ] Reconheço quando usar pseudo-inversa vs inversa

### Tópicos Críticos Revisitar

1. Multiplicação matricial (compatibilidade, associatividade, não-comutatividade)
2. Autovalores interpretação geométrica
3. SVD e relação com rank
4. Aplicações em ML (regressão, redes neurais, PCA)

## Proximos Passos

### Caminho Recomendado

1. **Praticar (hoje)**: Exercícios 1-3, erros comuns
2. **Aprofundar (próxima sessão)**: Calculus (0.4) para entender derivadas de matrizes
3. **Aplicar (depois)**: Regressão Linear (1.2) para usar em problemas reais
4. **Avançar (semana 2)**: PCA (3.1) e decomposições especializadas

### Recursos para Praticar

- Usar numpy array playground: experimente formas (shapes)- Visualizar transformações com matplotlib
- Implementar operações do zero antes de usar np.linalg

### Sinais de Compreensão✓ Sabe quando @ funciona (verifica shape antes)✓ Entende rank vs determinante vs invertibilidade✓ Pode explicar PCA com autovalores/autovetores✓ Sabe vantagens de SVD vs eigendecomposition

### Tópicos Avançados (Opcional)- Decomposição QR (ortogonalização)- Polardecomposição (A = UP)- Tensor decomposição (generalização para 3D+)- Álgebra linear numérica (estabilidade, condicionamento)

In [ ]:
# SOLUCAO - Exercício 1
import numpy as np

# Criar e multiplicar matrizes
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

# Multiplicação matricial
C = A @ B
print("A @ B =")
print(C)
print(f"Shape: {C.shape}")

# Verificar: (2,2) @ (2,2) = (2,2) ✓
assert C.shape == (2, 2)


In [ ]:
# SOLUCAO - Exercício 2
import numpy as np

# Determinar rank e invertibilidade
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

rank = np.linalg.matrix_rank(A)
det = np.linalg.det(A)

print(f"Rank: {rank}")
print(f"Determinante: {det:.2e}")
print(f"Invertível? {rank == 3}")

# Usar pseudo-inversa (funciona mesmo para singular)
A_pinv = np.linalg.pinv(A)
print(f"Pseudo-inversa shape: {A_pinv.shape}")


#### conexao com

- **Física e Simulação**: Matriz de massa, amortecimento, rigidez
- **Equações Diferenciais**: Discretização gera sistemas matriciais
- **Solvers**: Resolver Ax=b é central em simulação numérica

In [ ]:
# SOLUCAO - Exercício 3
import numpy as np

# SVD truncado para compressão
A = np.random.randn(10, 8)

# Decompor
U, s, Vt = np.linalg.svd(A, full_matrices=False)

# Truncar para k=3
k = 3
U_k = U[:, :k]
s_k = s[:k]
Vt_k = Vt[:k, :]

# Reconstruir
A_approx = U_k @ np.diag(s_k) @ Vt_k

# Erro
erro = np.linalg.norm(A - A_approx)
print(f"Erro Frobenius: {erro:.6f}")
print(f"Razão compressão: {A.size / (U_k.size + s_k.size + Vt_k.size):.2f}x")


#### conexao com

- **Graphs e Redes Sociais**: Matriz de adjacência representa grafo
- **PageRank**: Usa autovalores da matriz de transição
- **Community Detection**: SVD na matriz de adjacência

## Resumo

### Hierarquia de Conceitos

#### Nível 1: Fundamentos

- Tipos de matrizes (zero, identidade, diagonal, etc)- Operações básicas (soma, multiplicação escalar)

#### Nível 2: Operações Principais

- Multiplicação matricial A @ B- Determinante e rank
- Inversa de matrizes

#### Nível 3: Decomposição

- Autovalores e autovetores
- Eigendecomposition (A = QΛQ^-1)- SVD (A = UΣV^T)

#### Nível 4: Aplicações

- PCA para redução de dimensionalidade
- Compressão de imagens
- Recomendação colaborativa

### Checklist de Aprendizado- [ ] Entendo operações matriciais e requisitos de compatibilidade- [ ] Posso calcular multiplicação, determinante e rank manualmente- [ ] Entendo diferença entre autovalores e valores singulares- [ ] Posso aplicar SVD em problemas reais- [ ] Conheço conexões com regressão linear e redes neurais- [ ] Sei usar np.linalg para operações matriciais- [ ] Reconheço quando usar pseudo-inversa vs inversa

### Tópicos Críticos Revisitar

1. Multiplicação matricial (compatibilidade, associatividade, não-comutatividade)
2. Autovalores interpretação geométrica
3. SVD e relação com rank
4. Aplicações em ML (regressão, redes neurais, PCA)

## Proximos Passos

### Caminho Recomendado

1. **Praticar (hoje)**: Exercícios 1-3, erros comuns
2. **Aprofundar (próxima sessão)**: Calculus (0.4) para entender derivadas de matrizes
3. **Aplicar (depois)**: Regressão Linear (1.2) para usar em problemas reais
4. **Avançar (semana 2)**: PCA (3.1) e decomposições especializadas

### Recursos para Praticar

- Usar numpy array playground: experimente formas (shapes)- Visualizar transformações com matplotlib
- Implementar operações do zero antes de usar np.linalg

### Sinais de Compreensão✓ Sabe quando @ funciona (verifica shape antes)✓ Entende rank vs determinante vs invertibilidade✓ Pode explicar PCA com autovalores/autovetores✓ Sabe vantagens de SVD vs eigendecomposition

### Tópicos Avançados (Opcional)- Decomposição QR (ortogonalização)- Polardecomposição (A = UP)- Tensor decomposição (generalização para 3D+)- Álgebra linear numérica (estabilidade, condicionamento)